# 40 - Sim-To-Real For Humanoids

## What / Why / How

**What we are trying to do:** Study sim-to-real for humanoids: randomization, residual errors, hardware limits, and staged deployment.

**Why this matters:** Humanoid policies that work in sim can fail quickly on hardware because contacts, delays, and actuators differ.

**How we will do it:** Run randomized rollouts, quantify robustness, and build a staged deployment checklist.

## Humanoid Sim-To-Real

The question is not whether simulation is useful. It is where simulation lies, and how you protect the robot when reality disagrees.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(40)

def run_policy(randomized=True):
    mass_scale = rng.uniform(0.8, 1.25) if randomized else 1.0
    motor_delay = rng.integers(0, 5) if randomized else 1
    foot_friction = rng.uniform(0.45, 1.1) if randomized else 0.8
    command = 0.0
    buffer = [0.0] * motor_delay
    speed = 0.0
    distance = 0.0
    fall = False
    for step in range(160):
        desired = 0.8
        command = np.clip(2.0 * (desired - speed), -1.0, 1.0)
        buffer.append(command)
        delayed = buffer.pop(0)
        speed += (delayed / mass_scale - 0.05 * speed) * 0.02
        distance += speed * 0.02
        if abs(command) > foot_friction * 1.3:
            fall = True
            break
    return distance, speed, fall, {"mass_scale": mass_scale, "delay": motor_delay, "friction": foot_friction}

trials = [run_policy(True) for _ in range(300)]
fall_rate = np.mean([t[2] for t in trials])
distances = np.array([t[0] for t in trials])
print("fall rate:", fall_rate)
print("distance mean/p10:", distances.mean(), np.percentile(distances, 10))

In [ ]:
failures = [meta for _, _, fall, meta in trials if fall]
print("num failures:", len(failures))
print("first failures:", failures[:5])

In [ ]:
deployment_stages = [
    ("sim", "10k randomized episodes", "no regression in fall rate"),
    ("sim-to-sim", "different physics backend", "similar metrics"),
    ("bench", "legs suspended or low power", "commands bounded"),
    ("tethered", "overhead support", "no hard falls"),
    ("free walk", "clear test area", "human outside fall zone"),
]
for stage in deployment_stages:
    print(stage)

## Exercises

1. Add sensor noise and estimate state.
2. Add a safety filter that reduces command when friction is low.
3. Write a deployment stop condition for each stage.